In [ ]:
import pandas as pd
from tqdm.notebook import tqdm

## Setup

Load the raw roll call vote data (assuming it is clean - it is not currently but lets just pretend for now), list of bills voted on and the list of all representatives.

We only care about the list of bills because that gives us access to the general court number the vote took place which can help us when joining these tables.

In [ ]:
bill_df = pd.read_csv('../../data/parsedcsv/hjbills_clean.tsv', sep='\t')
vote_df = pd.read_csv('../../data/parsedcsv/hjrollcallvotes_raw.csv')
rep_df = pd.read_csv('../../data/parsedcsv/MAlegislators_GC182-194.csv')

In [ ]:
bill_df.head()

In [ ]:
vote_df.head()

In [ ]:
len(vote_df)

I filter down the DF of representatives to only those that serve in the House since we are not dealing with the Senate at all right now

In [ ]:
rep_df = rep_df[rep_df['branch'] == 'House']
rep_df.drop(inplace=True, columns=['Unnamed: 0'])
rep_df.head()

Checking whether our list of representatives covers the years of votes we have, we see that the list of representatives actually goes beyond the years we have of votes (2002 does appear in the Roll Call website, but does not have any roll call votes PDFs saved for it, so technically they should match but no data exists for 2002 despite it having a page for it.)

This means we should be covered and can successfully merge our rep dataset for all years of roll call votes.

In [ ]:
bill_df.general_court_number.unique()

In [ ]:
rep_df.court_number.unique()

Now I join the bill df with the vote df to attach the general court number to the vote so that only representatives who served in th

In [ ]:
joined_vote_bill_df = bill_df.merge(vote_df, how='inner', on='bill_id')[['general_court_number', 'bill_id', 'voter_name', 'vote_value', 'rollcall_date']]
joined_vote_bill_df.head()

### Minor data cleaning

We need to replace all instances of Mr. Speaker with the standing speaker of the house at each instance. This method is currently overgeneralized (Speakers rarely transitioned at the change of a GC), but for quick prototyping, I am making the following assumptions for now.

< 183 = Finneran

< 186 = DiMasi

< 191 = DeLeo

< 194 = Mariano

In [ ]:
joined_vote_bill_df.loc[((joined_vote_bill_df['voter_name'].str.contains('Speaker') == True) & (joined_vote_bill_df['general_court_number'] <= 183)), 'voter_name'] = 'Finneran'
joined_vote_bill_df.loc[((joined_vote_bill_df['voter_name'].str.contains('Speaker') == True) & (joined_vote_bill_df['general_court_number'] <= 186) & (joined_vote_bill_df['general_court_number'] > 183)), 'voter_name'] = 'DiMasi'
joined_vote_bill_df.loc[((joined_vote_bill_df['voter_name'].str.contains('Speaker') == True) & (joined_vote_bill_df['general_court_number'] <= 191) & (joined_vote_bill_df['general_court_number'] > 186)), 'voter_name'] = 'DeLeo'
joined_vote_bill_df.loc[((joined_vote_bill_df['voter_name'].str.contains('Speaker') == True) & (joined_vote_bill_df['general_court_number'] <= 194) & (joined_vote_bill_df['general_court_number'] > 191)), 'voter_name'] = 'DiMasi'

In [ ]:
joined_vote_bill_df[joined_vote_bill_df['voter_name'].str.contains('Speaker') == True]

Next, we need to also remove NaNs from the voter_name column of the votes. This occurs when there is a number of votes on a PDF not evenly divisible by 4 so blank spaces were picked up by PyMuPDF and added to the data.

In [ ]:
joined_vote_bill_df[joined_vote_bill_df['voter_name'].isna() == True]

In [ ]:
joined_vote_bill_df = joined_vote_bill_df.dropna(subset=['voter_name'])
joined_vote_bill_df[joined_vote_bill_df['voter_name'].isna() == True]

In [ ]:
print(f'{len(vote_df) - len(joined_vote_bill_df)} NaN rows removed from cleaned dataset')

Lastly, we need to handle some of the expected naming trends in the dataset. There are two major ones to focus on: the * which indicates that a vote was an 'after vote' and the -- -- dashes that indicates when the party changes in the roll call.

We want to preserve the after_vote information since that is relevant to our output, but we don't care about the party change dashes so we want to parse and strip these from names that have them.

In [ ]:
# Handling * in names

# Add column with True when name contains an *
joined_vote_bill_df['is_after_vote'] = joined_vote_bill_df['voter_name'].str.contains('\*')

# Strip * from all voter_names
joined_vote_bill_df['voter_name'] = joined_vote_bill_df['voter_name'].map(lambda x: x.lstrip('*').rstrip('*').replace('*', ''))

# Print cleaned dataframe
joined_vote_bill_df[joined_vote_bill_df['voter_name'].str.contains('\*') == True]

In [ ]:
# Handle -- -- 
joined_vote_bill_df['voter_name'] = joined_vote_bill_df['voter_name'].map(lambda x: x.lstrip('--').rstrip('--'))

joined_vote_bill_df[joined_vote_bill_df['voter_name'].str.contains('--') == True]

## Run Matching on Reps

### Test Match votes for a single general court

To test a matching function, I took a subset of the vote dataset and rep dataset based on the first General Court the vote dataset covers: 183.

In [ ]:
test_votes_df = joined_vote_bill_df[joined_vote_bill_df['general_court_number'] == 189]
test_votes_clean_df = test_votes_df.dropna(subset=['voter_name'])

In [ ]:
test_votes_clean_df

In [ ]:
test_reps_df = rep_df[rep_df['court_number'] == 189]

In [ ]:
test_reps_df

A function ```matchName(str, str)``` for matching name depending on format of the name string is defined below.

Critically, while a majority of names in the vote data is just the last name, in any instances where there exists a duplicate last name (more than 1 rep with the same last name) the vote data includes initials. Sometimes these initials are comma-separated or space-separated.

This function takes this into account and goes to the initial level when a match is made but the name from the vote data contains additional initials to avoid duplicates or incorrect matching.

In [ ]:
def matchName(name_to_match, name_to_check):

    # If name contains a comma, split on comma
    if ',' in name_to_match:
        split_name = name_to_match.split(',')
        last_name, first_names = split_name[0], split_name[1].strip()

        # Check if last name matches
        if last_name in name_to_check:
            first_initial = first_names[0]

            # Check if first initial matches first letter of full name
            if first_initial == name_to_check[0]:
                return True
            else:
                return False
        else:
            return False
    # Check if string has multiple names that are separated by spaces
    elif len(name_to_match.split(' ')) > 1 and name_to_match.split(' ')[1]:
        split_name = name_to_match.split(' ')
        last_name, first_names = split_name[0], split_name[1]

        # Check if last name matches
        if last_name in name_to_check:
            first_initial = first_names[0]

            # Check if first initial matches first letter of full name
            if first_initial == name_to_check[0]:
                return True
            else:
                return False
        else:
            return False
    else: # Name is just a single name
        name_to_match = name_to_match.strip() # Strip whitespace for edge cases

        if name_to_match in name_to_check:
            return True
        return False


By looping through all rows of the vote dataset subset and calling the matchName function, we can see how many matches are made. 

In [ ]:
verbose = False

# Iterate over all rows of test votes df
for i in test_votes_clean_df.index:

    name = test_votes_clean_df.at[i, 'voter_name']

    # Iterate over all rep names in subset
    for j in test_reps_df.index:
        check_name = test_reps_df.at[j, 'full_name']

        # Check if name is found in name to check
        if verbose: print(f'{i}: Checking {name} against {check_name}')
        if matchName(name,check_name):
            member_code = test_reps_df.at[j, 'member_code']
            if verbose: print(f'Found match!! Member Code: {member_code}')
            test_votes_clean_df.at[i, 'member_code'] = member_code
            break

In [ ]:
test_votes_clean_df[test_votes_clean_df['member_code'].isna() == True]

### Batch process matching across all General Court Years

Now that we've seen a pretty good result (still will need manual checking, since it appears that 3-5 names per GC are not coming up in the rep dataset), we can run it over all the GCs.

A function ```matchAllNames(pd.DataFrame, pd.DataFrame)``` is defined that iterates over the range of all GCs we are working with. It takes the input of dataframes containing all votes and all reps and does the same subsetting process for each GC. The matching function is called for each row and an output DataFrame containing the matched values is concatenated over the iterations.

At the end of the function, some summary statistics of the matching output are printed for clarity.

In [ ]:
%pip install --upgrade jupyter ipywidgets

In [ ]:
def matchAllNames(all_votes_df, all_reps_df):
    # Silence SettingsWithCopyWarning for this run
    pd.options.mode.chained_assignment = None

    # Track matches that failed
    failed_matches = 0

    # Create a DataFrame to return
    out_cols = all_votes_df.columns.to_list()
    out_cols.append('member_code')
    out_votes_df = pd.DataFrame(columns=out_cols)

    # Iterate over all General Courts contained in the votes data
    for gc in range(183, 195):
        print(f'Matching reps and votes from General Court #{gc}')

        # Get the votes and representatives for current General Court
        gc_votes_df = all_votes_df[joined_vote_bill_df['general_court_number'] == gc]
        gc_reps_df = all_reps_df[rep_df['court_number'] == gc]

        # Iterate over all rows of test votes df - use tqdm to show progress of current GC matching
        for i in tqdm(gc_votes_df.index):

            name = gc_votes_df.at[i, 'voter_name']

            # Iterate over all rep names in subset
            for j in gc_reps_df.index:
                check_name = gc_reps_df.at[j, 'full_name']

                # Check if name is found in name to check
                if verbose: print(f'{i}: Checking {name} against {check_name}')
                if matchName(name,check_name):
                    member_code = gc_reps_df.at[j, 'member_code']
                    if verbose: print(f'Found match!! Member Code: {member_code}')
                    gc_votes_df.at[i, 'member_code'] = member_code

                    # Break once a match is found since a member should not match more than once
                    break

        # Aggregate number of failed matches
        failed_matches += len(gc_votes_df[gc_votes_df['member_code'].isna() == True])

        # After matching for this GC, concat output to out_votes_df
        out_votes_df = pd.concat([out_votes_df, gc_votes_df])

    # After matching all General Courts
    print(f'Percentage of failed matches: {failed_matches} / {len(out_votes_df)} = {(failed_matches / len(out_votes_df)) * 100}% failure rate')

    return out_votes_df

In [ ]:
matches_votes_df = matchAllNames(joined_vote_bill_df, rep_df)

After matching is done, we see a pretty good result of about only a 10% failure rate. I know from my knowledge of the data that a majority of this comes from the 2009 - 2017 section of the data so this is something we will need to deal with directly.

In [ ]:
matches_votes_df[matches_votes_df['member_code'].isna() == True]['voter_name'].unique()

## Save output to csv

After matching is done, all we need to do is drop the non-matches (for now, will not drop the final product since we don't want to lose votes) and drop the voter_name column now that we have the MemberCode instead. 

In [ ]:
final_vote_df = matches_votes_df.dropna(subset=['member_code'])

In [ ]:
final_vote_df = final_vote_df[['bill_id', 'member_code', 'vote_value','is_after_vote']].reset_index()

In [ ]:
final_vote_df

In [ ]:
final_vote_df.drop(inplace=True, columns=['index'])

Lastly, we can write the output to a CSV file that will be saved to the project Google Drive.

In [ ]:
final_vote_df.to_csv('../../data/parsedcsv/hjrollcallvotes_cleanedv1.csv', index=False)

In [ ]:
test_read_df = pd.read_csv('../../data/parsedcsv/hjrollcallvotes_cleanedv1.csv')
test_read_df